In [1]:
from pyspark.sql import functions as F
from pyspark.sql import Window

from utils import Silver


required_bronze_tables = ['Green','Yellow']

silver = Silver('GY_Pre_validation', 'GY_Validated', 'GY_Invalidated', required_bronze_tables)

green_df = silver.bronze_tables_dfs['Green']
yellow_df = silver.bronze_tables_dfs['Yellow']

green_df = green_df.select(
    green_df.VendorID.alias('VendorId'),
    green_df.lpep_pickup_datetime.alias('PickUpDateTime'),
    green_df.lpep_dropoff_datetime.alias('DropOffDateTime'),
    green_df.PULocationID.alias('PickUpLocationId'),
    green_df.DOLocationID.alias('DropOffLocationId'),
    green_df.passenger_count.alias('PassengerCount'),
    green_df.trip_distance.alias('TripDistance'),
    green_df.tip_amount.alias('TipAmount'),
    green_df.total_amount.alias('TotalAmount')
)

yellow_df = yellow_df.select(
    yellow_df.VendorID.alias('VendorId'),
    yellow_df.tpep_pickup_datetime.alias('PickUpDateTime'),
    yellow_df.tpep_dropoff_datetime.alias('DropOffDateTime'),
    yellow_df.PULocationID.alias('PickUpLocationId'),
    yellow_df.DOLocationID.alias('DropOffLocationId'),
    yellow_df.passenger_count.alias('PassengerCount'),
    yellow_df.trip_distance.alias('TripDistance'),
    yellow_df.tip_amount.alias('TipAmount'),
    yellow_df.total_amount.alias('TotalAmount')
)

GY_Pre_validation_df = green_df.unionAll(yellow_df).na.fill('999',["VendorId"])

In [2]:
# We only want to dedup valid rows
GY_Validated_df = GY_Pre_validation_df\
    .filter((F.col('PassengerCount') > 0) | (F.col('PassengerCount').isNotNull()))

# Intial data analysis showed negative TotalAmounts this check tells us if TotalAmount is causing the dup values
window = Window.partitionBy('VendorId','PickUpDateTime','DropOffDateTime','PickUpLocationId','DropOffLocationId').orderBy(F.col('TotalAmount').desc())
GY_Validated_df = GY_Validated_df.withColumn('row', F.row_number().over(window))

In [8]:
# count number of duplicate values
dups_df = GY_Validated_df.filter(F.col('row') > 1)

dups_df.count() 

6103

In [12]:
# getting a sample
dups_df.orderBy('VendorId', 'PickUpDateTime', 'DropOffDateTime', 'PickUpLocationId', 'DropOffLocationId').filter(F.col('TotalAmount') < 0).show(1)

+--------+-------------------+-------------------+----------------+-----------------+--------------+------------+---------+-----------+---+
|VendorId|     PickUpDateTime|    DropOffDateTime|PickUpLocationId|DropOffLocationId|PassengerCount|TripDistance|TipAmount|TotalAmount|row|
+--------+-------------------+-------------------+----------------+-----------------+--------------+------------+---------+-----------+---+
|       2|2021-01-01 00:11:01|2021-01-01 00:13:56|             263|               75|             1|        1.13|        0|       -8.8|  2|
+--------+-------------------+-------------------+----------------+-----------------+--------------+------------+---------+-----------+---+
only showing top 1 row



In [13]:
# validate the sample has duplicate rows in original GY_Pre_validation_df
duplicates_removed_df = GY_Validated_df.filter(F.col('VendorId') == '2').filter(F.col('PickUpDateTime') == '2021-01-01 00:11:01')

duplicates_removed_df.show()

+--------+-------------------+-------------------+----------------+-----------------+--------------+------------+---------+-----------+---+
|VendorId|     PickUpDateTime|    DropOffDateTime|PickUpLocationId|DropOffLocationId|PassengerCount|TripDistance|TipAmount|TotalAmount|row|
+--------+-------------------+-------------------+----------------+-----------------+--------------+------------+---------+-----------+---+
|       2|2021-01-01 00:11:01|2021-01-01 00:13:56|             263|               75|             1|        1.13|        0|        8.8|  1|
|       2|2021-01-01 00:11:01|2021-01-01 00:13:56|             263|               75|             1|        1.13|        0|       -8.8|  2|
+--------+-------------------+-------------------+----------------+-----------------+--------------+------------+---------+-----------+---+



In [3]:
# if no duplicates found, count should be zero
duplicates_removed_df = GY_Validated_df.filter(F.col('row') == 1)

duplicate_check_df = duplicates_removed_df.groupBy(
        F.col('VendorId'),
        F.col('PickUpDateTime'),
        F.col('DropOffDateTime'),
        F.col('PickUpLocationId'),
        F.col('DropOffLocationId'),
).agg(F.count('*').alias('RowCount'))

duplicate_check_df = duplicate_check_df.filter(F.col('RowCount') > 1)

duplicate_check_df.count()

0